In [ ]:
# Example 1: Build an Atoms object and inspect basic properties.
from ase import Atoms
import numpy as np

# Water molecule
water = Atoms(
    'H2O',
    positions=[[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 0.5]],
)

print(f"Chemical formula: {water.get_chemical_formula()}")
print(f"Number of atoms: {len(water)}")
print(f"Atomic numbers: {water.get_atomic_numbers()}")
print(f"Positions shape: {water.get_positions().shape}")

assert len(water) == 3
assert water.get_chemical_formula() == 'H2O'
assert water.get_atomic_numbers().tolist() == [1, 1, 8]


In [ ]:
# Example 2: Use ase.build to create well-known structures.
from ase.build import bulk, molecule

# Bulk crystal (copper, FCC)
cu = bulk('Cu', 'fcc', a=3.6)
print(f"Cu bulk: {cu.get_chemical_formula()}, cell volume = {cu.get_volume():.3f}")
print(f"Cu cell:\n{cu.cell.array}")

# Molecule from the G2 database
h2o = molecule('H2O')
print(f"\nH2O molecule: {h2o.get_chemical_formula()}")
print(f"Distances:\n{h2o.get_all_distances()}")

assert cu.get_chemical_formula() == 'Cu'
assert h2o.get_chemical_formula() == 'H2O'
assert cu.get_volume() > 0


In [ ]:
# Example 3: Periodic cell + supercell construction.
from ase.build import bulk

si = bulk('Si', 'diamond', a=5.43)
print(f"Primitive Si cell: {len(si)} atoms, volume = {si.get_volume():.3f}")
print(f"PBC: {si.pbc}")

# Build a 2x2x2 supercell
super_si = si * (2, 2, 2)
print(f"Supercell: {len(super_si)} atoms, volume = {super_si.get_volume():.3f}")

assert len(super_si) == 8 * len(si)
assert abs(super_si.get_volume() - 8 * si.get_volume()) < 1e-6
assert all(super_si.pbc)


In [ ]:
# Example 4: I/O round-trip via the XYZ format.
import io
from ase.build import molecule
from ase.io import read, write

co2 = molecule('CO2')

buf = io.StringIO()
write(buf, co2, format='xyz')
buf.seek(0)

co2_back = read(buf, format='xyz')
print(f"Original: {co2.get_chemical_formula()}, {len(co2)} atoms")
print(f"Round-trip: {co2_back.get_chemical_formula()}, {len(co2_back)} atoms")

assert co2_back.get_chemical_formula() == co2.get_chemical_formula()
assert len(co2_back) == len(co2)
assert (co2_back.get_atomic_numbers() == co2.get_atomic_numbers()).all()


In [ ]:
# Example 5: Attach a calculator and evaluate energy + forces.
from ase.build import molecule
from ase.calculators.emt import EMT

# EMT supports Ni, Cu, Pd, Ag, Pt, Au, Al, H, C, N, O
atoms = molecule('H2')
atoms.calc = EMT()

energy = atoms.get_potential_energy()
forces = atoms.get_forces()

print(f"H2 potential energy: {energy:.6f} eV")
print(f"H2 forces shape: {forces.shape}")
print(f"H2 forces:\n{forces}")

import numpy as np
assert isinstance(energy, float)
assert forces.shape == (2, 3)
# Newton's third law: forces should sum (approximately) to zero
assert np.allclose(forces.sum(axis=0), 0.0, atol=1e-8)
